In [ ]:
# Imports

import librosa
import scipy.spatial.distance as distance
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import csv
import matplotlib.backends.backend_pdf
import scipy.signal as sig
import functions as func

In [ ]:
# Short-time analysis parameters

fs = 16000
t_STFT_frame = 0.04  # for a frame width of 0.04s
t_STFT_hop = 0.005  # for a hop of 0.005 s
t_ACF_lag = 1.5 # ACF calculated upto 3 seconds
t_ACF_frame = 3  # 10 seconds window over which ACF is calculated
t_ACF_hop = 0.5  # successive ACF frames are shifted by 0.5 s

In [ ]:
# Ground truth structural boundaries
ground_truth_boundaries=[0, 302, 357, 375, 415, 442, 491, 567, 644, 729, 797, 840]
ground_truth_cycles=[0, 61, 73, 77, 86, 91, 103, 120, 140, 160, 178, 190]
ground_truth_colors=['k', 'r', 'b', 'r', 'b', 'brown', 'b', 'r', 'b', 'r', 'b', 'k']
ground_truth_labels=['V', 'L', 'V', 'L', 'V', 'T', 'V', 'L', 'V', 'L', 'V']
tempo_change_boundaries=[67, 127, 657]
GT = ground_truth_boundaries

In [ ]:
# Load audio
file_path = '../audios/instrumental_solo/SA_AmAK_BAH_Jhaptal.wav'
audio,_ = librosa.load(file_path,sr=fs)
audio /= np.max(np.abs(audio))

actual_total_time = float(len(audio))/fs
fs_hop = int(1/t_STFT_hop)

In [ ]:
# Compute STFT
S, P = func.STFT(audio, fs, t_STFT_hop, t_STFT_frame)
(l, N) = np.shape(S)
offset = float(N)/fs
time = np.arange(offset, offset + l*t_STFT_hop, t_STFT_hop)

In [ ]:
# Compute P-ODF which is sensitve to onsets of both melodic instrument and tabla
ODF_all = np.zeros(l)
for i in range(l):
    ODF_all[i] = np.sum(S[i] > S[i-1])    
ODF_all = ODF_all - np.mean(ODF_all)
ODF_all = ODF_all/np.max(ODF_all)

In [ ]:
# Compute tabla-sensitive P-T-ODF
ODF_tabla = np.copy(ODF_all)
ODF_tabla[ODF_tabla > -0.3] = -0.3
ODF_tabla = -ODF_tabla - 0.3

In [ ]:
# Compute ACF for onsets of each of the above types
ACF_all,_ = func.ACF_DFT(ODF_all, t_ACF_lag, t_ACF_frame, t_ACF_hop, fs_hop)
ACF_tabla,_ = func.ACF_DFT(ODF_tabla, t_ACF_lag, t_ACF_frame, t_ACF_hop, fs_hop)

T, n_lag = np.shape(ACF_all)
time = np.arange(0, actual_total_time, t_ACF_hop)
time = time[:len(ACF_all)]
total_time = time[-1]

In [ ]:
# All onsets Rhythmogram (by P-ODF)
ACF_all[ACF_all<0] = 0
log_ACF_all = np.log(ACF_all+1e-10)
log_ACF_all[log_ACF_all <= -4.606] = -10
log_ACF_all[np.isnan(log_ACF_all)] = -10
log_ACF_all = log_ACF_all + 10
log_ACF_all[:,:7] = np.zeros((T, 7))
smooth_ACF_all = func.rhythmogram_smoothing(log_ACF_all)

In [ ]:
# Tabla centric rhythmogram (by P-T-ODF)
ACF_tabla[ACF_tabla<0] = 0
log_ACF_tabla = np.log(ACF_tabla+1e-10)
log_ACF_tabla[log_ACF_tabla <= -2.303] = -10
log_ACF_tabla[np.isnan(log_ACF_tabla)] = -10
log_ACF_tabla = log_ACF_tabla + 10
log_ACF_tabla[:,:7] = np.zeros((T, 7))
smooth_ACF_tabla = func.rhythmogram_smoothing(log_ACF_tabla)

In [ ]:
# Display both rhythmograms
func.display_ACF(smooth_ACF_all, total_time, t_ACF_lag, GT, take_log=0, heading='Overall Rhythmogram')
func.display_ACF(smooth_ACF_tabla, total_time, t_ACF_lag, GT, take_log=0, heading='Tabla Rhythmogram')

In [ ]:
kernel = func.kernel_generator(25)
plt.imshow(kernel, origin='lower', aspect='auto')
plt.colorbar()

In [ ]:
# Compute SDM and novelty 
ACF_window = np.ones(n_lag)
ACF_window[fs_hop//2:3*fs_hop//2] = np.linspace(1, 0, fs_hop*1)

windowed_ACF_all = smooth_ACF_all*ACF_window
windowed_ACF_tabla = smooth_ACF_tabla*ACF_window

SDM_all = distance.squareform(distance.pdist(windowed_ACF_all, 'cosine'))
novelty_function_all = func.novelty_function_generator(SDM_all, kernel, time)

SDM_tabla = distance.squareform(distance.pdist(windowed_ACF_tabla, 'cosine'))
novelty_function_tabla = func.novelty_function_generator(SDM_tabla, kernel, time)

In [ ]:
# Display SDM and novelty function
func.display_SDM_novelty(SDM_all, novelty_function_all, [], time, GT, tempo_change_boundaries)
func.display_SDM_novelty(SDM_tabla, novelty_function_tabla, [], time, GT, tempo_change_boundaries)